In [ ]:
# transferable code functions
import numpy as np
import scipy.signal as scisig


def padding(layer:np.ndarray, mode:str = 'zero', pad_size:int = 1):

    for i in range(pad_size):

        padded_ly_size = (layer.shape[0]+2, layer.shape[1]+2)
        padded_ly = np.zeros(padded_ly_size)
        padded_ly[1:-1,1:-1] = layer
    
        if mode == 'same':
            padded_ly[0,1:-1] = layer[0,:]
            padded_ly[-1,1:-1] = layer[-1,:]

            padded_ly[1:-1,0] = layer[:,0]
            padded_ly[1:-1,-1] = layer[:,-1]

            padded_ly[0,0] = layer[0,0]
            padded_ly[0,-1] = layer[0,-1]
            padded_ly[-1,0] = layer[-1,0]
            padded_ly[-1,-1] = layer[-1,-1]
        layer = padded_ly

    return(padded_ly)

def convolve(input:np.ndarray, kernel:np.ndarray, pad_mode:str = 'zero', pad_size:int = 666, conv_mode:str="valid"):
    if pad_mode !="none":
        if pad_size == 666:
            pad_size = kernel.shape[0]//2
        input = padding(input, pad_mode, pad_size)
    k_shp = kernel.shape
    #print(input.shape)
    result = scisig.convolve(input, kernel, mode=conv_mode, method="fft")

    return(result)

def patch_convolve(input:np.ndarray, patch_shape:tuple, error_array:np.ndarray, pad_mode:str = 'zero', pad_size:int = 666, conv_mode:str="valid"):
    if pad_mode !="none":
        if pad_size == 666:
            pad_size = patch_shape[0]//2
        input = padding(input, pad_mode, pad_size)
    
    error_patch_arr = np.zeros(shape=(*error_array.shape, *patch_shape))

    for i in range(error_array.shape[0]):
        for j in range(error_array.shape[1]):
            error_patch_arr[i,j,:,:] = input[i:i+patch_shape[0], j:j+patch_shape[1]] * error_array[i,j]
    kernel_delta = np.sum(error_patch_arr, axis=(0,1))
    return(kernel_delta)

def fast_patch_convolve(input:np.ndarray, patch_shape:tuple, error_array:np.ndarray, 
                        pad_mode:str = 'zero', pad_size:int = 666, conv_mode:str="valid"):
    from numpy.lib.stride_tricks import sliding_window_view
    if pad_mode !="none":
        if pad_size == 666:
            pad_size = patch_shape[0]//2
        input = padding(input, pad_mode, pad_size)

    patches = sliding_window_view(input, (3,3))
    op1 = patches * error_array[:,:, np.newaxis, np.newaxis] #Need to understand this better

    dW = np.sum(op1, axis=(0,1))
    #dw is the delta in the filter/kernel/patch
    return(dW)

def pool(input:np.ndarray, stride = 2, mode = "max"):
    shape = (int(input.shape[0]/stride), int(input.shape[1]/stride))
    output = np.zeros(shape)
    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            if mode =="max":
                output[i,j] = np.max(input[i*stride:i*stride+stride,j*stride:j*stride+stride])
                #look at that absolute index fuckery
            if mode =="mean":
                output[i,j] = np.mean(input[i*stride:i*stride+stride,j*stride:j*stride+stride])
    return(output)

def sigmond(input):
    return( 1/(1+np.exp(-input)))

def sigmond_prime(input):
    return(sigmond(input) * (1-sigmond(input)))

def ReLU(input):
    return(np.maximum(0, input))

def ReLU_prime(input):
    return(input > 0).astype(input.dtype)

def deep_copy_mat_list (list_to_copy, propigate_cell_vals=True):
    #creates a copy of a list containing numpy arrays of varible size, ethier as empty arrays or with the same values
    
    new_list = []
    if propigate_cell_vals:
        for items in list_to_copy:
            new_list.append(items)
    else:
        for items in list_to_copy:
            new_list.append(np.zeros_like(items))       

    return(new_list)


In [ ]:
import numpy as np
from omegaconf import DictConfig, OmegaConf

def label_to_output(int) -> np.ndarray:
    outlayer = np.zeros(shape=(10), dtype=np.float32)
    outlayer[int] = 1
    return(outlayer)

def label_vec_to_output(inputs: np.ndarray) -> np.ndarray:
    # Create an identity matrix of size 10 and index into it
    return np.eye(10, dtype=np.int8)[inputs]

def create_batch(trn_cfg, image_file, label_file):
    """
    Given a hyper parameter cfg object, a loaded image file, and label file, 
    reads from both files and returns:

    input_batch: an ndarray of shape (batch_size, image_size_height, image_size_width) of type float32
    
    label_batch: an ndarray of shape (batch_size, possible_labels) of type int64
    """
    image_buffer = image_file.read(trn_cfg.image_size[0] * trn_cfg.image_size[1] * trn_cfg.batch_size)
    image_batch = np.frombuffer(image_buffer, dtype=np.uint8).astype(np.float32)
    image_batch = np.reshape(image_batch, shape = (trn_cfg.batch_size, 
                                                    trn_cfg.image_size[0], 
                                                    trn_cfg.image_size[1]))
    label_buffer = label_file.read(trn_cfg.batch_size)
    label_pre_batch = np.frombuffer(label_buffer, dtype=np.uint8).astype(np.int64)
    label_batch = label_vec_to_output(label_pre_batch)

    return(image_batch, label_batch)


def loss(network_output_ly, expected_ly):
    #loss =  mean of all output neurons - the expected, squared
    loss_vec = ((network_output_ly - expected_ly)**2) / 2
    loss_sclr = np.sum((network_output_ly - expected_ly)**2)
    return(loss_sclr, loss_vec)

#todo I'd like to create a version of this that can have dynamically sized convolutional layers
import hydra
from omegaconf import DictConfig, OmegaConf
mdl_cfg = OmegaConf.load("model.yaml")
import numpy as np
from dataclasses import dataclass
from omegaconf import DictConfig
import scipy.signal as scisig


def build_index_lookup(cfg: DictConfig):
    """
    Given a loaded OmegaConf config with a 'layers' section,
    build a lookup table: index -> (layer_name, layer_data).
    """
    blocks = cfg.blocks

    index_lookup = {
        block_data.index: (block_name, block_data)
        for block_name, block_data in blocks.items()
    }
    return index_lookup


class Input_block:
    def __init__(self, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index

        self.b_type = "input"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)

    def backprop(self, expected:np.ndarray=None):
        return(0)

class FC_block:
    def __init__(self, prev_ly_shape, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index
        self.prev_ly_shape = prev_ly_shape

        self.b_type = "fc"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)
        self.biases = np.zeros(layer_shape)
        self.weights = self._init_weights()
        self.W_delta = np.zeros_like(self.weights)        
        self.B_delta = np.zeros_like(self.biases)
        self.fc_weights_shape = self.weights.shape


    def _init_weights(self, init=True):
        if init:
            weights = np.random.uniform(-1,1, size=(*self.prev_ly_shape, *self.layer_shape))
        else:
            weights = np.zeros(shape=(*self.prev_ly_shape, *self.layer_shape))
        return(weights)
    
    def forward(self, input):
        if len(input.shape) == 1:
            self.z_values = np.tensordot(input, self.weights, axes=((0),(0)))  + self.biases
        if len(input.shape) == 2:
            self.z_values = np.tensordot(input, self.weights, axes=((0,1),(0,1)))  + self.biases
        self.activations = ReLU(self.z_values)

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        if expected is not None:
            del_l = (expected - self.activations) * ReLU_prime(self.z_values)

        #print("weights shape: ", self.weights.shape)
        #print("current layer shape: ", self.layer_shape)
        #print("previous layer shape: ", self.prev_ly_shape)
        #print("del_l shape: ", del_l.shape)
        if len(self.prev_ly_shape) == 1:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((1),(0)))# * prior_z_vals
        if len(self.prev_ly_shape) == 2:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((2),(0)))# * prior_z_vals

        self.W_delta = np.tensordot(prior_activations, del_l, axes=0)
        self.B_delta = del_l

        #print("\nweights shape: ", self.weights.shape)
        #print("weights Delta shape: ", self.W_delta.shape)
        return(del_l_neg1, self.W_delta, self.B_delta)

class FC_CONV_block:
    def __init__(self, prev_bk_shape, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index
        self.prev_bk_shape = prev_bk_shape

        self.b_type = "fc_conv"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)
        self.biases = np.zeros(layer_shape)
        self.weights = self._init_weights()
        self.fc_weights_shape = self.weights.shape

    def _init_weights(self, init=True):
        if init:
            weights = np.random.uniform(-1,1, size=(*self.prev_bk_shape, *self.layer_shape))
        else:
            weights = np.zeros(shape=(*self.prev_bk_shape, *self.layer_shape))
        return(weights)
    
    def forward(self, input):
        self.z_values = np.tensordot(input, self.weights, axes=((0,1,2),(0,1,2)))
        self.activations = ReLU(self.z_values)

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        if expected is not None:
            del_l = (expected - self.activations) * ReLU_prime(self.z_values)

        #print("weights shape: ", self.weights.shape)
        #print("current layer shape: ", self.layer_shape)
        #print("previous layer shape: ", self.prev_bk_shape)
        #print("del_l shape: ", del_l.shape)
        
        if len(self.layer_shape) == 2:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((3,4),(0,1)))# * prior_z_vals
        if len(self.layer_shape) == 1:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((3),(0)))# * prior_z_vals
        self.W_delta = np.tensordot(prior_activations, del_l, axes=0)
        self.B_delta = del_l

        #print("\nweights shape: ", self.weights.shape)
        #print("weights Delta shape: ", self.W_delta.shape)
        return(del_l_neg1, self.W_delta, self.B_delta)

class Conv_Block:
    def __init__(self, prev_bk_depth, num_filters, kernel_shape, stride, layer_shape, index):
        self.num_filters = num_filters
        self.kernel_shape = kernel_shape
        self.stride = stride
        self.layer_shape = layer_shape
        self.index = index
        self.prev_bk_depth = prev_bk_depth


        self.b_type = "conv"
        self.ly_dim = len(layer_shape)
        self.filters = self.init_filters(self.prev_bk_depth, self.num_filters, self.kernel_shape, init = True) #are of size (h,m,3,3) 

        self.z_values = np.zeros(shape=(self.num_filters, *self.layer_shape))
        self.biases = np.zeros(shape=(self.num_filters,)) 
        self.activations = np.zeros_like(self.z_values)

        #self.submaps = np.zeros(shape=(self.prev_bk_depth, self.num_filters, *self.layer_shape))
        self.W_delta = np.zeros_like(self.filters)        
        self.B_delta = np.zeros_like(self.biases)#not sure about this

    def _init_feature_maps(self):
        feature_maps = np.zeros(shape=(self.num_filters, *self.layer_shape))
        feature_map_z_vals = np.zeros_like(feature_maps)  
        return(feature_maps, feature_map_z_vals)

    @staticmethod    
    def init_filters(prior_num_filters:int, num_filters:int, kernel_shape:tuple, init=True):
        if init:
            filters = np.random.uniform(-1,1, size=(prior_num_filters, num_filters, *kernel_shape))
        else:
            filters = np.zeros((prior_num_filters, num_filters, *kernel_shape))
        return (filters)
    
    def OLD_forward(self, input:np.ndarray):
        #input should be an ndarray of h num_channels i hieght j width
        for m in range(self.feature_maps.shape[0]):
            if len(input.shape) == 2:
                input = np.expand_dims(input, axis=0)
            temp_maps = np.zeros_like(input)
            for h in range(input.shape[0]):
                temp_maps[h, :, :] = convolve(input[h,:,:], self.filters[m,:,:], conv_mode="valid")
            self.submaps[:,m,:,:] = temp_maps
            self.feature_map_z_vals[m,:,:] = np.sum(temp_maps, axis=0) + self.feature_map_biases[m, :,:]
        self.feature_maps = ReLU(self.feature_map_z_vals)
        self.activations = self.feature_maps
        self.z_values = self.feature_map_z_vals

    def forward(self, input:np.ndarray):
        if len(input.shape) == 2:
            input = np.expand_dims(input, axis=0)
        for m in range(self.filters.shape[1]):
            for h in range(self.filters.shape[0]):
                self.z_values[m,:,:] += convolve(input[h,:,:], self.filters[h,m,:,:], conv_mode="valid")
            self.z_values[m,:,:] += self.biases[m]
        self.activations = ReLU(self.z_values)
        return(self.activations)

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
            if len(prior_z_vals.shape) == 2:
                prior_z_vals = np.expand_dims(prior_z_vals, axis=0)
                #might not need this code
            if len(prior_activations.shape) == 2:
                prior_activations = np.expand_dims(prior_activations, axis=0)
                #might not need this code

            prior_z_prime = ReLU_prime(prior_z_vals)
            neg1_error = np.zeros_like(prior_z_vals)

            for m in range(self.filters.shape[1]):
                #find bias changes:
                self.B_delta[m] = np.sum(del_l[m])

                for h in range(self.filters.shape[0]):
                    #backprop error:
                    kernel = self.filters[h, m]
                    flipped_kernel = np.flip(kernel, axis=(0, 1))

                    back = scisig.correlate2d(del_l[m], flipped_kernel, mode="full")
                    trimmed = back[1:-1, 1:-1]
                    neg1_error[h] += trimmed

                    #find dW:
                    self.W_delta[h,m,:,:] = scisig.convolve2d(prior_activations[h,:,:], del_l[m,:,:], mode="valid")
                
            
            del_l_neg1 = neg1_error * prior_z_prime
            #print("l-1 ERROR shape: ", del_l_neg1.shape)
            return(del_l_neg1, self.W_delta, self.B_delta)


class Pooling_ly:
    def __init__(self, shape, num_filters, index, stride:int=2, p_type:str="max"):
        self.index = index
        self.shape = shape
        self.depth = num_filters
        self.stride = stride
        self.p_type = p_type
        self.b_type = "pooling"


        self.activations = np.ndarray(shape=(self.depth, *self.shape))
        self.z_values = np.zeros_like(self.activations)
        self.weights = np.ndarray(shape=(1,))
        self.biases = np.ndarray(shape=(1,))


    def pooling(self, input_act):
        if len(input_act.shape) == 2:
            input_act = np.expand_dims(input_act, axis=0)
        
        self.activations = np.zeros((input_act.shape[0], *self.shape))
        for i in range(input_act.shape[0]):
            self.activations[i,:,:] = pool(input_act[i,:,:], stride=self.stride, mode=self.p_type)
        

    def forward(self, input):
        self.pooling(input)
        self.z_values = self.activations

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        #print("pooling del_l input shape: ", del_l.shape)
        if expected is not None:
            del_l = (expected - self.activations) * (self.activations) #Suspect

        #print("current layer shape: ", self.shape)
        #print("del_l shape: ", del_l.shape)
        
        del_l_neg1 = np.repeat(np.repeat(del_l, 2, axis=1), 2, axis=2) #should be expanded over axes 1,2. axis 0 is the feature map dim
        #print("pooling del_l next layer back shape:", del_l_neg1.shape)
        #MIGHT NEED TO CORRECT THIS TO ONLY PROPIGATE ERROR TO THE LOCATIONS WHICH TRIGGERED MAX POOL
        self.W_delta = np.zeros_like(self.activations)        
        self.B_delta = np.zeros_like(self.activations)

        return(del_l_neg1, self.W_delta, self.B_delta)

class NN:
    def __init__(self, config:DictConfig, blocks: list):
        self.config = config
        self.blocks = blocks

    @classmethod
    def create_network(cls, cfg:DictConfig, **kwargs):
        index_to_block = build_index_lookup(cfg)
        print("creating network....")

        blocks = []
        for items in cfg.blocks:
            #print(items)
            block_name, block_data = index_to_block[cfg.blocks[items].index]
            

            if block_data.type == "input":
                blocks.append(Input_block(block_data.shape, block_data.index))
            if block_data.type == "fc":
                if prev_bk_data.type == "conv2D":
                    prev_bk_shp = (prev_bk_data.filters.filter_num, *prev_bk_data.shape)
                    blocks.append(FC_CONV_block(prev_bk_shp, block_data.shape, block_data.index))
                else:
                    blocks.append(FC_block(prev_bk_data.shape, block_data.shape, block_data.index))
            if block_data.type == "pool":
                blocks.append(Pooling_ly(block_data.shape, prev_bk_data.filters.filter_num, block_data.index, block_data.stride, block_data.mode))

            if block_data.type == "conv2D":
                fltr = block_data.filters
                if prev_bk_data.type == "conv2D":
                    prev_bk_dpth = prev_bk_data.filters.filter_num
                elif prev_bk_data.type == "pool":
                    prev_bk_dpth = blocks[-1].depth
                else:
                    prev_bk_dpth = 1

                blocks.append(Conv_Block(prev_bk_dpth, fltr.filter_num, fltr.kernel_shape, fltr.stride, block_data.shape, block_data.index))

            prev_bk_name = block_name
            prev_bk_data = block_data

        print("Done!")
        return cls(cfg, blocks)
    
    def forward(self, input_ly):
        print("running forward function....")
        self.blocks[0].activations = input_ly
        self.blocks[0].z_values = input_ly
        for index in range(len(self.blocks)):
            #print("\nindex: ", index)
            if index == 0:
                self.blocks[0].activations = input_ly
            else:
                self.blocks[index].forward(self.blocks[index-1].activations)
        print("Done!\n")
            

    def backprop(self, expected):
        #print("running backprop....")
        dW = []
        dB = []
        for index in range(len(self.blocks)-1, 0 , -1):


            #print("index: ", index)
            if index == len(self.blocks)-1:
                delL_back1, dW_tmp, dB_tmp = self.blocks[index].backprop(prior_activations = self.blocks[index-1].activations, 
                                                         prior_z_vals= self.blocks[index-1].z_values,
                                                         expected = expected
                                                         )
            else:
                delL_back1, dW_tmp, dB_tmp = self.blocks[index].backprop(prior_activations = self.blocks[index-1].activations, 
                                                         prior_z_vals= self.blocks[index-1].z_values,
                                                         del_l = delL_back1
                                                         )
            if self.blocks[index].b_type =="pooling":
                pass
            else:
                dW.append(dW_tmp)
                dB.append(dB_tmp)
        dW.reverse()
        dB.reverse()#doing this because they were created from back to front
        #print("Done!")
        return(dW, dB)

def batch_backprop(input_batch:np.ndarray, desired_output_batch:np.ndarray,
                    trn_cfg:DictConfig, mdl_cfg:DictConfig, n_net:NN):
    batch_dW = []
    batch_dB = []
    batch_loss = np.ndarray((trn_cfg.batch_size,))

    for blocks in n_net.blocks:
        if blocks.b_type == "input":
            pass
        elif blocks.b_type == "pooling":
            pass
        else:
            if blocks.b_type == "conv":
                weights_shape = blocks.filters.shape
            else:
                weights_shape = blocks.weights.shape
            bias_shape = blocks.biases.shape

            batch_dW.append(np.zeros( shape=(trn_cfg.batch_size, *weights_shape)))
            batch_dB.append(np.zeros( shape=(trn_cfg.batch_size, *bias_shape)))


    for single_sample in range(trn_cfg.batch_size):        
        print("sample/batch: ", single_sample)
        n_net.forward(input_batch[single_sample])
        dW, dB = n_net.backprop(desired_output_batch[single_sample])
        


        #print(len(n_net.blocks))
        #print(len(dW))
        sub_index = 0
        for index, blocks in enumerate(n_net.blocks):
            #print(index)
            #print(sub_index)
            if blocks.b_type == "input":
                pass
            elif blocks.b_type == "pooling":
                pass
            else:
                batch_dW[sub_index][single_sample] = dW[sub_index]
                batch_dB[sub_index][single_sample] = dB[sub_index]
                sub_index = sub_index+1


        loss_sclr, loss_vec = loss(n_net.blocks[-1].activations, desired_output_batch[single_sample])
        print("sample loss: ", loss_sclr)
        batch_loss[single_sample] = loss_sclr

        if single_sample == 50:
            print("convoltional block 2 activations: ", n_net.blocks[3].activations)
            print("output activations: ", n_net.blocks[-1].activations)

    mean_loss = np.mean(batch_loss)
    print("batch loss: ",mean_loss)
    mean_dW = []
    mean_dB = []
    for blocks in n_net.blocks:
        if blocks.b_type == "input":
            pass
        else:
            if blocks.b_type == "conv":
                weights_shape = blocks.filters.shape
            else:
                weights_shape = blocks.weights.shape
            bias_shape = blocks.biases.shape
            mean_dW.append(np.zeros(shape=(weights_shape)))
            mean_dB.append(np.zeros(shape=(bias_shape)))
        
    sub_index = 0
    for index, blocks in enumerate(n_net.blocks):
            print(index)
            print(sub_index)
            if blocks.b_type == "input":
                pass
            elif blocks.b_type == "pooling":
                pass
            else:
                mean_dW[sub_index] = np.mean(batch_dW[sub_index], axis=0) * trn_cfg.learning_rate #learning rate is here!
                mean_dB[sub_index] = np.mean(batch_dB[sub_index], axis=0) * trn_cfg.learning_rate #learning rate is here!
                sub_index = sub_index+1
    #for idx in range(len(n_net.blocks) -1):
    #    mean_dW[idx] = np.mean(batch_dW[idx], axis=0) * trn_cfg.learning_rate #learning rate is here!
    #    mean_dB[idx] = np.mean(batch_dB[idx], axis=0) * trn_cfg.learning_rate #learning rate is here!


    return(mean_dW, mean_dB, mean_loss)

def train(trn_cfg:DictConfig, mdl_cfg:DictConfig, model):
    """
    this is the core training loop that trains through epochs.  
    requires training_cfg(trn_cfg) containing hyperparameters and image and label paths,
    and model_cfg(mdl_cfg) containing model architecture.

    """
    import time
    import matplotlib.pyplot as plt


    x_data, y_data = [],[]
    from IPython.display import clear_output
    start = time.time()

    for epochs in range(trn_cfg.epochs):
        print("epoch ", int(epochs), "out of ", int(trn_cfg.epochs))
        f = open(trn_cfg.images_path, 'rb')
        f.read(16)
        l = open(trn_cfg.labels_path,'rb')
        l.read(8)
        for batches in range(0, trn_cfg.dataset_size, trn_cfg.batch_size):
            images, labels = create_batch(trn_cfg, f,l)

            batch_dW, batch_dB, batch_loss = batch_backprop(images, labels, trn_cfg, mdl_cfg, model)
            sub_index = 0
            for index, blocks in enumerate(model.blocks):
                    print(index)
                    print(sub_index)
                    if blocks.b_type == "input":
                        pass
                    elif blocks.b_type == "pooling":
                        pass
                    else:
                        if blocks.b_type == "conv":
                            model.blocks[index].filters = model.blocks[index].filters + batch_dW[sub_index]
                        else:
                            model.blocks[index].weights = model.blocks[index].weights + batch_dW[sub_index]
                        model.blocks[index].biases = model.blocks[index].biases + batch_dB[sub_index]
                        sub_index = sub_index+1

            #create batches of batch size, feed forward, backprop, save dW, dB temporayly, avereage accross batch,
            #apply averaged dW, dB to network, repeat for all batchs
        #repeat for all epochs

        x_data.append(epochs)
        y_data.append(batch_loss)
        clear_output(wait=True)        # Clear previous plot
        plt.figure(figsize=(6, 4))
        plt.plot(x_data, y_data, 'bo-')
        plt.title("Epoch vs Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss at epoch end")
        plt.grid(True)
        plt.show()
    end = time.time()
    run_time = int(end - start)


    return(model, run_time, (x_data, y_data))

#todo:
    #figure out network autocreation DONE!
    #figure out padding algorrithm DONE!
    #figure out down sizing  DONE!
        #this will be through pooling 
    #write forward functions DONE!
        #since these differ between different types of layers and blocks, 
        # maybe each block should have a forward function?
    #write backprop DONE!
        #now done with symbolic backprop creation, need to consider:
        #should there be a backprop function for each block/layer or a golbal
    #revaluate model architechure for practical ability to detect objects

    #make a visualizer
    
    #figure out what's causing NAN values
    #figure out memory problem
    #make more efficient



In [ ]:
#todo I'd like to create a version of this that can have dynamically sized convolutional layers
import hydra
from omegaconf import DictConfig, OmegaConf
mdl_cfg = OmegaConf.load("model.yaml")
import numpy as np
from dataclasses import dataclass
from omegaconf import DictConfig
import scipy.signal as scisig


def build_index_lookup(cfg: DictConfig):
    """
    Given a loaded OmegaConf config with a 'layers' section,
    build a lookup table: index -> (layer_name, layer_data).
    """
    blocks = cfg.blocks

    index_lookup = {
        block_data.index: (block_name, block_data)
        for block_name, block_data in blocks.items()
    }
    return index_lookup


class Input_block:
    def __init__(self, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index

        self.b_type = "input"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)

    def backprop(self, expected:np.ndarray=None):
        return(0)

class FC_block:
    def __init__(self, prev_ly_shape, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index
        self.prev_ly_shape = prev_ly_shape

        self.b_type = "fc"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)
        self.biases = np.zeros(layer_shape)
        self.weights = self._init_weights()
        self.W_delta = np.zeros_like(self.weights)        
        self.B_delta = np.zeros_like(self.biases)
        self.fc_weights_shape = self.weights.shape


    def _init_weights(self, init=True):
        if init:
            weights = np.random.uniform(-1,1, size=(*self.prev_ly_shape, *self.layer_shape))
        else:
            weights = np.zeros(shape=(*self.prev_ly_shape, *self.layer_shape))
        return(weights)
    
    def forward(self, input):
        if len(input.shape) == 1:
            self.z_values = np.tensordot(input, self.weights, axes=((0),(0)))  + self.biases
        if len(input.shape) == 2:
            self.z_values = np.tensordot(input, self.weights, axes=((0,1),(0,1)))  + self.biases
        self.activations = ReLU(self.z_values)

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        if expected is not None:
            del_l = (expected - self.activations) * ReLU_prime(self.z_values)

        #print("weights shape: ", self.weights.shape)
        #print("current layer shape: ", self.layer_shape)
        #print("previous layer shape: ", self.prev_ly_shape)
        #print("del_l shape: ", del_l.shape)
        if len(self.prev_ly_shape) == 1:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((1),(0)))# * prior_z_vals
        if len(self.prev_ly_shape) == 2:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((2),(0)))# * prior_z_vals

        self.W_delta = np.tensordot(prior_activations, del_l, axes=0)
        self.B_delta = del_l

        #print("\nweights shape: ", self.weights.shape)
        #print("weights Delta shape: ", self.W_delta.shape)
        return(del_l_neg1, self.W_delta, self.B_delta)

class FC_CONV_block:
    def __init__(self, prev_bk_shape, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index
        self.prev_bk_shape = prev_bk_shape

        self.b_type = "fc_conv"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)
        self.biases = np.zeros(layer_shape)
        self.weights = self._init_weights()
        self.fc_weights_shape = self.weights.shape

    def _init_weights(self, init=True):
        if init:
            weights = np.random.uniform(-1,1, size=(*self.prev_bk_shape, *self.layer_shape))
        else:
            weights = np.zeros(shape=(*self.prev_bk_shape, *self.layer_shape))
        return(weights)
    
    def forward(self, input):
        self.z_values = np.tensordot(input, self.weights, axes=((0,1,2),(0,1,2)))
        self.activations = ReLU(self.z_values)

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        if expected is not None:
            del_l = (expected - self.activations) * ReLU_prime(self.z_values)

        #print("weights shape: ", self.weights.shape)
        #print("current layer shape: ", self.layer_shape)
        #print("previous layer shape: ", self.prev_bk_shape)
        #print("del_l shape: ", del_l.shape)
        
        if len(self.layer_shape) == 2:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((3,4),(0,1)))# * prior_z_vals
        if len(self.layer_shape) == 1:
            del_l_neg1 = np.tensordot(self.weights, del_l, axes=((3),(0)))# * prior_z_vals
        self.W_delta = np.tensordot(prior_activations, del_l, axes=0)
        self.B_delta = del_l

        #print("\nweights shape: ", self.weights.shape)
        #print("weights Delta shape: ", self.W_delta.shape)
        return(del_l_neg1, self.W_delta, self.B_delta)

class Conv_Block:
    def __init__(self, prev_bk_depth, num_filters, kernel_shape, stride, layer_shape, index):
        self.num_filters = num_filters
        self.kernel_shape = kernel_shape
        self.stride = stride
        self.layer_shape = layer_shape
        self.index = index
        self.prev_bk_depth = prev_bk_depth


        self.b_type = "conv"
        self.ly_dim = len(layer_shape)
        self.filters = self.init_filters(self.prev_bk_depth, self.num_filters, self.kernel_shape, init = True) #are of size (h,m,3,3) 

        self.z_values = np.zeros(shape=(self.num_filters, *self.layer_shape))
        self.biases = np.zeros(shape=(self.num_filters,)) 
        self.activations = np.zeros_like(self.z_values)

        #self.submaps = np.zeros(shape=(self.prev_bk_depth, self.num_filters, *self.layer_shape))
        self.W_delta = np.zeros_like(self.filters)        
        self.B_delta = np.zeros_like(self.biases)#not sure about this

    def _init_feature_maps(self):
        feature_maps = np.zeros(shape=(self.num_filters, *self.layer_shape))
        feature_map_z_vals = np.zeros_like(feature_maps)  
        return(feature_maps, feature_map_z_vals)

    @staticmethod    
    def init_filters(prior_num_filters:int, num_filters:int, kernel_shape:tuple, init=True):
        if init:
            filters = np.random.uniform(-1,1, size=(prior_num_filters, num_filters, *kernel_shape))
        else:
            filters = np.zeros((prior_num_filters, num_filters, *kernel_shape))
        return (filters)
    
    def OLD_forward(self, input:np.ndarray):
        #input should be an ndarray of h num_channels i hieght j width
        for m in range(self.feature_maps.shape[0]):
            if len(input.shape) == 2:
                input = np.expand_dims(input, axis=0)
            temp_maps = np.zeros_like(input)
            for h in range(input.shape[0]):
                temp_maps[h, :, :] = convolve(input[h,:,:], self.filters[m,:,:], conv_mode="valid")
            self.submaps[:,m,:,:] = temp_maps
            self.feature_map_z_vals[m,:,:] = np.sum(temp_maps, axis=0) + self.feature_map_biases[m, :,:]
        self.feature_maps = ReLU(self.feature_map_z_vals)
        self.activations = self.feature_maps
        self.z_values = self.feature_map_z_vals

    def forward(self, input:np.ndarray):
        if len(input.shape) == 2:
            input = np.expand_dims(input, axis=0)
        for m in range(self.filters.shape[1]):
            for h in range(self.filters.shape[0]):
                self.z_values[m,:,:] += convolve(input[h,:,:], self.filters[h,m,:,:], conv_mode="valid")
            self.z_values[m,:,:] += self.biases[m]
        self.activations = ReLU(self.z_values)
        return(self.activations)

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
            if len(prior_z_vals.shape) == 2:
                prior_z_vals = np.expand_dims(prior_z_vals, axis=0)
                #might not need this code
            if len(prior_activations.shape) == 2:
                prior_activations = np.expand_dims(prior_activations, axis=0)
                #might not need this code

            prior_z_prime = ReLU_prime(prior_z_vals)
            neg1_error = np.zeros_like(prior_z_vals)

            for m in range(self.filters.shape[1]):
                #find bias changes:
                self.B_delta[m] = np.sum(del_l[m])

                for h in range(self.filters.shape[0]):
                    #backprop error:
                    kernel = self.filters[h, m]
                    flipped_kernel = np.flip(kernel, axis=(0, 1))

                    back = scisig.correlate2d(del_l[m], flipped_kernel, mode="full")
                    trimmed = back[1:-1, 1:-1]
                    neg1_error[h] += trimmed

                    #find dW:
                    self.W_delta[h,m,:,:] = scisig.convolve2d(prior_activations[h,:,:], del_l[m,:,:], mode="valid")
                
            
            del_l_neg1 = neg1_error * prior_z_prime
            #print("l-1 ERROR shape: ", del_l_neg1.shape)
            return(del_l_neg1, self.W_delta, self.B_delta)


class Pooling_ly:
    def __init__(self, shape, num_filters, index, stride:int=2, p_type:str="max"):
        self.index = index
        self.shape = shape
        self.depth = num_filters
        self.stride = stride
        self.p_type = p_type
        self.b_type = "pooling"


        self.activations = np.ndarray(shape=(self.depth, *self.shape))
        self.z_values = np.zeros_like(self.activations)
        self.weights = np.ndarray(shape=(1,))
        self.biases = np.ndarray(shape=(1,))


    def pooling(self, input_act):
        if len(input_act.shape) == 2:
            input_act = np.expand_dims(input_act, axis=0)
        
        self.activations = np.zeros((input_act.shape[0], *self.shape))
        for i in range(input_act.shape[0]):
            self.activations[i,:,:] = pool(input_act[i,:,:], stride=self.stride, mode=self.p_type)
        

    def forward(self, input):
        self.pooling(input)
        self.z_values = self.activations

    def backprop(self, prior_activations:np.ndarray=None, prior_z_vals:np.ndarray=None, del_l:np.ndarray=None, expected:np.ndarray=None):
        #print("pooling del_l input shape: ", del_l.shape)
        if expected is not None:
            del_l = (expected - self.activations) * (self.activations) #Suspect

        #print("current layer shape: ", self.shape)
        #print("del_l shape: ", del_l.shape)
        
        del_l_neg1 = np.repeat(np.repeat(del_l, 2, axis=1), 2, axis=2) #should be expanded over axes 1,2. axis 0 is the feature map dim
        #print("pooling del_l next layer back shape:", del_l_neg1.shape)
        #MIGHT NEED TO CORRECT THIS TO ONLY PROPIGATE ERROR TO THE LOCATIONS WHICH TRIGGERED MAX POOL
        self.W_delta = np.zeros_like(self.activations)        
        self.B_delta = np.zeros_like(self.activations)

        return(del_l_neg1, self.W_delta, self.B_delta)

class NN:
    def __init__(self, config:DictConfig, blocks: list):
        self.config = config
        self.blocks = blocks

    @classmethod
    def create_network(cls, cfg:DictConfig, **kwargs):
        index_to_block = build_index_lookup(cfg)
        print("creating network....")

        blocks = []
        for items in cfg.blocks:
            #print(items)
            block_name, block_data = index_to_block[cfg.blocks[items].index]
            

            if block_data.type == "input":
                blocks.append(Input_block(block_data.shape, block_data.index))
            if block_data.type == "fc":
                if prev_bk_data.type == "conv2D":
                    prev_bk_shp = (prev_bk_data.filters.filter_num, *prev_bk_data.shape)
                    blocks.append(FC_CONV_block(prev_bk_shp, block_data.shape, block_data.index))
                else:
                    blocks.append(FC_block(prev_bk_data.shape, block_data.shape, block_data.index))
            if block_data.type == "pool":
                blocks.append(Pooling_ly(block_data.shape, prev_bk_data.filters.filter_num, block_data.index, block_data.stride, block_data.mode))

            if block_data.type == "conv2D":
                fltr = block_data.filters
                if prev_bk_data.type == "conv2D":
                    prev_bk_dpth = prev_bk_data.filters.filter_num
                elif prev_bk_data.type == "pool":
                    prev_bk_dpth = blocks[-1].depth
                else:
                    prev_bk_dpth = 1

                blocks.append(Conv_Block(prev_bk_dpth, fltr.filter_num, fltr.kernel_shape, fltr.stride, block_data.shape, block_data.index))

            prev_bk_name = block_name
            prev_bk_data = block_data

        print("Done!")
        return cls(cfg, blocks)
    
    def forward(self, input_ly):
        print("running forward function....")
        self.blocks[0].activations = input_ly
        self.blocks[0].z_values = input_ly
        for index in range(len(self.blocks)):
            #print("\nindex: ", index)
            if index == 0:
                self.blocks[0].activations = input_ly
            else:
                self.blocks[index].forward(self.blocks[index-1].activations)
        print("Done!\n")
            

    def backprop(self, expected):
        #print("running backprop....")
        dW = []
        dB = []
        for index in range(len(self.blocks)-1, 0 , -1):


            #print("index: ", index)
            if index == len(self.blocks)-1:
                delL_back1, dW_tmp, dB_tmp = self.blocks[index].backprop(prior_activations = self.blocks[index-1].activations, 
                                                         prior_z_vals= self.blocks[index-1].z_values,
                                                         expected = expected
                                                         )
            else:
                delL_back1, dW_tmp, dB_tmp = self.blocks[index].backprop(prior_activations = self.blocks[index-1].activations, 
                                                         prior_z_vals= self.blocks[index-1].z_values,
                                                         del_l = delL_back1
                                                         )
            if self.blocks[index].b_type =="pooling":
                pass
            else:
                dW.append(dW_tmp)
                dB.append(dB_tmp)
        dW.reverse()
        dB.reverse()#doing this because they were created from back to front
        #print("Done!")
        return(dW, dB)

def batch_backprop(input_batch:np.ndarray, desired_output_batch:np.ndarray,
                    trn_cfg:DictConfig, mdl_cfg:DictConfig, n_net:NN):
    batch_dW = []
    batch_dB = []
    batch_loss = np.ndarray((trn_cfg.batch_size,))

    for blocks in n_net.blocks:
        if blocks.b_type == "input":
            pass
        elif blocks.b_type == "pooling":
            pass
        else:
            if blocks.b_type == "conv":
                weights_shape = blocks.filters.shape
            else:
                weights_shape = blocks.weights.shape
            bias_shape = blocks.biases.shape

            batch_dW.append(np.zeros( shape=(trn_cfg.batch_size, *weights_shape)))
            batch_dB.append(np.zeros( shape=(trn_cfg.batch_size, *bias_shape)))


    for single_sample in range(trn_cfg.batch_size):        
        print("sample/batch: ", single_sample)
        n_net.forward(input_batch[single_sample])
        dW, dB = n_net.backprop(desired_output_batch[single_sample])
        


        #print(len(n_net.blocks))
        #print(len(dW))
        sub_index = 0
        for index, blocks in enumerate(n_net.blocks):
            #print(index)
            #print(sub_index)
            if blocks.b_type == "input":
                pass
            elif blocks.b_type == "pooling":
                pass
            else:
                batch_dW[sub_index][single_sample] = dW[sub_index]
                batch_dB[sub_index][single_sample] = dB[sub_index]
                sub_index = sub_index+1


        loss_sclr, loss_vec = loss(n_net.blocks[-1].activations, desired_output_batch[single_sample])
        print("sample loss: ", loss_sclr)
        batch_loss[single_sample] = loss_sclr

        if single_sample == 50:
            print("convoltional block 2 activations: ", n_net.blocks[3].activations)
            print("output activations: ", n_net.blocks[-1].activations)

    mean_loss = np.mean(batch_loss)
    print("batch loss: ",mean_loss)
    mean_dW = []
    mean_dB = []
    for blocks in n_net.blocks:
        if blocks.b_type == "input":
            pass
        else:
            if blocks.b_type == "conv":
                weights_shape = blocks.filters.shape
            else:
                weights_shape = blocks.weights.shape
            bias_shape = blocks.biases.shape
            mean_dW.append(np.zeros(shape=(weights_shape)))
            mean_dB.append(np.zeros(shape=(bias_shape)))
        
    sub_index = 0
    for index, blocks in enumerate(n_net.blocks):
            print(index)
            print(sub_index)
            if blocks.b_type == "input":
                pass
            elif blocks.b_type == "pooling":
                pass
            else:
                mean_dW[sub_index] = np.mean(batch_dW[sub_index], axis=0) * trn_cfg.learning_rate #learning rate is here!
                mean_dB[sub_index] = np.mean(batch_dB[sub_index], axis=0) * trn_cfg.learning_rate #learning rate is here!
                sub_index = sub_index+1
    #for idx in range(len(n_net.blocks) -1):
    #    mean_dW[idx] = np.mean(batch_dW[idx], axis=0) * trn_cfg.learning_rate #learning rate is here!
    #    mean_dB[idx] = np.mean(batch_dB[idx], axis=0) * trn_cfg.learning_rate #learning rate is here!


    return(mean_dW, mean_dB, mean_loss)

def train(trn_cfg:DictConfig, mdl_cfg:DictConfig, model):
    """
    this is the core training loop that trains through epochs.  
    requires training_cfg(trn_cfg) containing hyperparameters and image and label paths,
    and model_cfg(mdl_cfg) containing model architecture.

    """
    import time
    import matplotlib.pyplot as plt


    x_data, y_data = [],[]
    from IPython.display import clear_output
    start = time.time()

    for epochs in range(trn_cfg.epochs):
        print("epoch ", int(epochs), "out of ", int(trn_cfg.epochs))
        f = open(trn_cfg.images_path, 'rb')
        f.read(16)
        l = open(trn_cfg.labels_path,'rb')
        l.read(8)
        for batches in range(0, trn_cfg.dataset_size, trn_cfg.batch_size):
            images, labels = create_batch(trn_cfg, f,l)

            batch_dW, batch_dB, batch_loss = batch_backprop(images, labels, trn_cfg, mdl_cfg, model)
            sub_index = 0
            for index, blocks in enumerate(model.blocks):
                    print(index)
                    print(sub_index)
                    if blocks.b_type == "input":
                        pass
                    elif blocks.b_type == "pooling":
                        pass
                    else:
                        if blocks.b_type == "conv":
                            model.blocks[index].filters = model.blocks[index].filters + batch_dW[sub_index]
                        else:
                            model.blocks[index].weights = model.blocks[index].weights + batch_dW[sub_index]
                        model.blocks[index].biases = model.blocks[index].biases + batch_dB[sub_index]
                        sub_index = sub_index+1

            #create batches of batch size, feed forward, backprop, save dW, dB temporayly, avereage accross batch,
            #apply averaged dW, dB to network, repeat for all batchs
        #repeat for all epochs

        x_data.append(epochs)
        y_data.append(batch_loss)
        clear_output(wait=True)        # Clear previous plot
        plt.figure(figsize=(6, 4))
        plt.plot(x_data, y_data, 'bo-')
        plt.title("Epoch vs Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss at epoch end")
        plt.grid(True)
        plt.show()
    end = time.time()
    run_time = int(end - start)


    return(model, run_time, (x_data, y_data))

#todo:
    #figure out network autocreation DONE!
    #figure out padding algorrithm DONE!
    #figure out down sizing  DONE!
        #this will be through pooling 
    #write forward functions DONE!
        #since these differ between different types of layers and blocks, 
        # maybe each block should have a forward function?
    #write backprop DONE!
        #now done with symbolic backprop creation, need to consider:
        #should there be a backprop function for each block/layer or a golbal
    #revaluate model architechure for practical ability to detect objects

    #make a visualizer
    
    #figure out what's causing NAN values
    #figure out memory problem
    #make more efficient



In [ ]:
mdl_cfg = OmegaConf.load("model.yaml")
lrn_cfg = OmegaConf.load("hyper_params.yaml")

model = NN.create_network(mdl_cfg)
train(lrn_cfg, mdl_cfg, model)

In [ ]:
from omegaconf import DictConfig, OmegaConf
lrn_cfg = OmegaConf.load("hyper_params.yaml")

f = open(lrn_cfg.images_path, 'rb')
f.read(16)
l = open(lrn_cfg.labels_path,'rb')
l.read(8)

image_batch, label_pre_batch = create_batch(lrn_cfg, f, l)
#print(image_batch[1,:,:], image_batch.shape)
#print(label_batch[1,:], label_batch.shape)
lrn_cfg.epochs = 1000
print(lrn_cfg.epochs)
lrn_cfg = OmegaConf.load("hyper_params.yaml")
print(lrn_cfg.epochs)


In [ ]:
mdl_cfg = OmegaConf.load("model.yaml")
lrn_cfg = OmegaConf.load("hyper_params.yaml")

input_ly = np.random.rand(28,28)
model = NN.create_network(mdl_cfg)
model.forward(input_ly)
#train(lrn_cfg, mdl_cfg, model)
#input_ly = np.random.rand(28,28)
#model.forward(input_ly)
expected  = np.zeros((10,))
expected[3] = 1

#mdl = model.blocks
#for blocks in mdl:
#    if blocks.b_type == "conv":
#        print("SUBMAPS shape: ",blocks.submaps.shape)

dW, dB = model.backprop(expected)
#print(len(dW), type(dW), len(dB), type(dB))



In [ ]:
print(model.blocks[4].W_delta.shape)
print(model.blocks[4].W_delta)


In [ ]:
item = model.blocks
print("blocks")
for items in item:
    print(items.activations.shape, items.index, items.b_type)
    name = f"block_{items.index}_Activations"
    globals()[name] = items.activations




In [ ]:
mdl_cfg = OmegaConf.load("model.yaml")
print(type(mdl_cfg))
print(mdl_cfg)
examine_1 = mdl_cfg.model.layers.input_ly.shape
print("\nexamine_1: ")
print(examine_1)
print(type(examine_1))
examine_2 = tuple(mdl_cfg.model.layers.input_ly.shape)
print("\nexamine_2: ")
print(examine_2)
print(type(examine_2))
#next to figure out how to handle .yaml files and DictConfig files

#net = NN.create_network(cfg)


In [ ]:
import numpy as np
filters  = {"prev_ly_size":28,
            "kernel_size": 3,
            "stride" : 1}
l1 = np.zeros(shape=(filters["prev_ly_size"],))
l11 = np.zeros(shape=(filters["prev_ly_size"],))

l2 = []
l1[0] = 1
for index, values in enumerate(l1):
    if index%filters["stride"] == 0:
        l1[index] = 1


    if l1[index] == 1:
        for x in range(filters["kernel_size"]): 
            y=x+1
            try:
                l11[index + (y - filters["kernel_size"]//2)] = l11[index + (y - filters["kernel_size"]//2)] + 1
            except IndexError as error:
                print("had an index error, continuing")
print(l1,"\n",l11)